In [1]:
from vllm import LLM, SamplingParams
from factowl.fact_validation import validate_facts
from transformers import AutoTokenizer
import os
import pandas as pd
from factowl.fact_validation import validate_facts

INFO 01-26 09:06:23 [__init__.py:244] Automatically detected platform cuda.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
%load_ext autoreload
%autoreload 2
%cd /my_dir/factowl/
!pip install -e ./
!pip -q install wikipedia jieba
%cd factowl/

In [3]:
!nvidia-smi

Mon Jan 26 09:06:33 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.274.02             Driver Version: 535.274.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        Off | 00000000:41:00.0 Off |                  N/A |
|  0%   27C    P0             113W / 370W |      0MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
model_name = 'meta-llama/Meta-Llama-3-8B-Instruct'


tokenizer = AutoTokenizer.from_pretrained(model_name)
vllm_model = LLM(
    model=model_name,
    # dtype="half",
    trust_remote_code=True,
    # gpu_memory_utilization=0.8,
)

INFO 01-26 09:06:57 [config.py:823] This model supports multiple tasks: {'embed', 'score', 'classify', 'reward', 'generate'}. Defaulting to 'generate'.
INFO 01-26 09:07:08 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 01-26 09:07:12 [core.py:455] Waiting for init message from front-end.
INFO 01-26 09:07:12 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Meta-Llama-3-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespa

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 01-26 09:07:18 [default_loader.py:272] Loading weights took 3.23 seconds
INFO 01-26 09:07:18 [gpu_model_runner.py:1624] Model loading took 14.9596 GiB and 4.261699 seconds
INFO 01-26 09:07:26 [backends.py:462] Using cache directory: /root/.cache/vllm/torch_compile_cache/feaf175a96/rank_0_0 for vLLM's torch.compile
INFO 01-26 09:07:26 [backends.py:472] Dynamo bytecode transform time: 6.85 s
INFO 01-26 09:07:32 [backends.py:135] Directly load the compiled graph(s) for shape None from the cache, took 5.703 s
INFO 01-26 09:07:33 [monitor.py:34] torch.compile takes 6.85 s in total
INFO 01-26 09:07:36 [gpu_worker.py:227] Available KV cache memory: 5.13 GiB
INFO 01-26 09:07:36 [kv_cache_utils.py:715] GPU KV cache size: 42,032 tokens
INFO 01-26 09:07:36 [kv_cache_utils.py:719] Maximum concurrency for 8,192 tokens per request: 5.13x
INFO 01-26 09:08:04 [gpu_model_runner.py:2048] Graph capturing finished in 28 secs, took 1.59 GiB
INFO 01-26 09:08:04 [core.py:171] init engine (profile, creat

In [5]:
 # Prepare sampling parameters
vllm_sampling_params = SamplingParams(
    temperature=0.9,
    max_tokens=8192,
)

In [6]:
base_input_dir = './'
base_out_long_dir = "./filtered_facts_long_output"
base_out_dir = "./llm_filtered_facts"
bad_out_dir = "./llm_bad_facts/"
if not os.path.exists(base_out_dir):
    os.makedirs(base_out_dir)
if not os.path.exists(bad_out_dir):
    os.makedirs(bad_out_dir)
if not os.path.exists(base_out_long_dir):
    os.makedirs(base_out_long_dir)

## LLM-based irrelevant fact filtration

In [9]:
# LLM-based atomic fact filtration
for fname in os.listdir(base_input_dir):
    if not fname.endswith('tsv'):
        continue
    inp_p = os.path.join(base_input_dir, fname)
    out_long_p = os.path.join(base_out_long_dir, fname)
    out_p = os.path.join(base_out_dir, fname)
    bad_p = os.path.join(bad_out_dir, fname)

    labeled_df = validate_facts(inp_p, out_long_p, out_p, vllm_model, tokenizer,
                                vllm_sampling_params, claim_column='atom', num_examples=10)
    labeled_df["is_good"] = labeled_df["is_good"].astype(bool)
    bad_df = labeled_df[~labeled_df["is_good"]]
    bad_df["sample_id"] = list(range(bad_df.shape[0]))
    bad_df[["sample_id","topic","atom","is_supported","label"]].to_csv(bad_p, sep='\t', index=False)

Creating prompts...


100%|██████████| 41025/41025 [00:09<00:00, 4292.20it/s]

Generating responses for 41025 valid claims...


Adding requests:   0%|          | 0/41025 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/41025 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

...

Processed prompts:   0%|          | 0/32610 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

Processing outputs...
Saving long results to ./filtered_facts_long_output/pred_cars-gpt-5-chat_retrieval+llama_eval-wikipedia_api-p1-c5-eval-Qwen2.5-32B-Instruct.tsv


/my_dir/factowl/factowl/fact_validation.py:184: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_df["sample_id"] = list(range(clean_df.shape[0]))



SUMMARY
Total rows: 32610
Processed: 32610
Skipped (NaN): 0
GOOD: 31503 (96.6%)
BAD: 1107 (3.4%)
Long output saved to: ./filtered_facts_long_output/pred_cars-gpt-5-chat_retrieval+llama_eval-wikipedia_api-p1-c5-eval-Qwen2.5-32B-Instruct.tsv
Clean output saved to: ./filtered_facts/pred_cars-gpt-5-chat_retrieval+llama_eval-wikipedia_api-p1-c5-eval-Qwen2.5-32B-Instruct.tsv


/tmp/ipykernel_3528/3624128099.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bad_df["sample_id"] = list(range(bad_df.shape[0]))


## Filtration of facts not mentioning target entity

In [ ]:
base_out_dir_w_trg_ent = "./filtered_facts_w_target_entity"
bad_facts_out_dir = "./bad_facts_no_target_entity"
if not os.path.exists(base_out_dir_w_trg_ent):
    os.makedirs(base_out_dir_w_trg_ent)
if not os.path.exists(bad_facts_out_dir):
    os.makedirs(bad_facts_out_dir)
for fname in os.listdir(base_input_dir):
    if not fname.endswith('tsv'):
        continue
    in_p = os.path.join(base_out_dir, fname)
    out_p2 = os.path.join(base_out_dir_w_trg_ent, fname)
    bad_p = os.path.join(bad_facts_out_dir, fname)

    df = pd.read_csv(in_p, sep='\t')
    print(f"Path: {out_p}")
    print(f"BEFORE: {df.shape}")
    # Target entity must be mentioned in an atomic fact explicitly
    df["keep_flag"] = df.apply(lambda row: str(row["topic"]) in row["atom"], axis=1)
    good_df = df[df["keep_flag"]]
    bad_df = df[~df["keep_flag"]]

    good_df[["sample_id", "topic", "atom", "is_supported", "label"]].to_csv(out_p2,
                                                                       sep='\t',
                                                                      index=False)
    bad_df[["sample_id", "topic", "atom", "is_supported", "label"]].to_csv(bad_p,
                                                                       sep='\t',
                                                                      index=False)
    print(f"AFTER: {good_df.shape}")
    print(f"Bad facts: {bad_df.shape}")
    print('---')